In [2]:
!pip install ultralytics -q
print("Da cai dat xong ultralytics")

Da cai dat xong ultralytics


In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
import os

# ===== MODEL =====
# Baseline COCO (giai doan 1) - dung lam diem khoi tao cho lo trinh A
BASELINE_COCO = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/YOLOv8n/baseline_100epochs_es/weights/best.pt"

# Model cuoi cung cua lo trinh B (COCO->CanTho->HCMC) - da co san, dung de doi chieu
MODEL_SEQ_HCMC = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/finetune_hcmc_701515/weights/best.pt"

# ===== DATASET (goc tren Drive) =====
DRIVE_DIR_HCMC = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515"
DRIVE_DIR_COCO = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO_701515"
DATA_YAML_CANTHO = "/content/drive/MyDrive/Do an AI giao thong/Dataset/CanTho_701515/data.yaml"

# ===== DATASET (copy ve local de chay nhanh, on dinh) =====
LOCAL_DIR_HCMC = "/content/hcmc_701515"
LOCAL_DIR_COCO = "/content/coco_701515"

# ===== THU MUC LUU KET QUA =====
PROJECT_DIR = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline"
NAME_ABLATION = "ablation_finetune_hcmc_from_coco"   # ten thu muc luu model lo trinh A

# ===== CAU HINH HUAN LUYEN (dung Bang 3.4 - dam bao cong bang) =====
IMGSZ = 640
BATCH = 16
EPOCHS = 100
PATIENCE = 20
LR0 = 0.001
SEED = 42

print("Kiem tra su ton tai cac thanh phan:")
print("  Baseline COCO      :", os.path.exists(BASELINE_COCO))
print("  Model tuan tu HCMC :", os.path.exists(MODEL_SEQ_HCMC))
print("  Dataset HCMC (Drive):", os.path.exists(DRIVE_DIR_HCMC))
print("  Dataset COCO (Drive):", os.path.exists(DRIVE_DIR_COCO))
print("  Yaml CanTho         :", os.path.exists(DATA_YAML_CANTHO))

Kiem tra su ton tai cac thanh phan:
  Baseline COCO      : True
  Model tuan tu HCMC : True
  Dataset HCMC (Drive): True
  Dataset COCO (Drive): True
  Yaml CanTho         : True


In [5]:
import shutil, time

def copy_dataset_an_toan(drive_dir, local_dir, ten):
    if os.path.exists(local_dir):
        shutil.rmtree(local_dir)
    t0 = time.time()
    try:
        shutil.copytree(drive_dir, local_dir)
        print(f"[OK] Da copy {ten} ve {local_dir} ({time.time()-t0:.1f}s)")
    except shutil.Error as e:
        print(f"[CANH BAO] {len(e.args[0])} file loi tam thoi khi copy {ten}, kiem tra lai so luong sau.")

copy_dataset_an_toan(DRIVE_DIR_HCMC, LOCAL_DIR_HCMC, "HCMC")
copy_dataset_an_toan(DRIVE_DIR_COCO, LOCAL_DIR_COCO, "Vehicles-COCO")

DATA_YAML_HCMC_LOCAL = f"{LOCAL_DIR_HCMC}/data.yaml"
DATA_YAML_COCO_LOCAL = f"{LOCAL_DIR_COCO}/data.yaml"
print()
print("Yaml HCMC (local) ton tai:", os.path.exists(DATA_YAML_HCMC_LOCAL))
print("Yaml COCO (local) ton tai:", os.path.exists(DATA_YAML_COCO_LOCAL))

[OK] Da copy HCMC ve /content/hcmc_701515 (106.7s)
[OK] Da copy Vehicles-COCO ve /content/coco_701515 (608.4s)

Yaml HCMC (local) ton tai: True
Yaml COCO (local) ton tai: True


In [6]:
from ultralytics import YOLO

model_A = YOLO(BASELINE_COCO)   # khoi tao tu Baseline COCO

results_A = model_A.train(
    data=DATA_YAML_HCMC_LOCAL,
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMGSZ,
    batch=BATCH,
    lr0=LR0,
    seed=SEED,
    project=PROJECT_DIR,
    name=NAME_ABLATION,
    exist_ok=True,
    verbose=True,
)
print("\nDa huan luyen xong lo trinh A (COCO -> HCMC).")
BEST_A = os.path.join(PROJECT_DIR, NAME_ABLATION, "weights", "best.pt")
print("Model tot nhat luu tai:", BEST_A)
print("Ton tai:", os.path.exists(BEST_A))

Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hcmc_701515/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/YOLOv8n/baseline_100epochs_es/weights/b

KeyboardInterrupt: 

In [7]:
from ultralytics import YOLO

# Nap last.pt (KHONG phai best.pt) - vi last.pt chua trang thai day du
# de tiep tuc train, con best.pt chi la trong so tot nhat, khong co
# thong tin epoch/optimizer de resume
LAST_A = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/ablation_finetune_hcmc_from_coco/weights/last.pt"

model = YOLO(LAST_A)
model.train(resume=True)

Exception in thread Thread-6 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
    ~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.13/multiprocessing/queues.py", line 120, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
  File "/usr/lib/python3.13/multiprocessing/resource

Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hcmc_701515/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/ablation_finetune_hcmc_from_coco/weight

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c911e5e1390>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [8]:
BEST_A = os.path.join(PROJECT_DIR, NAME_ABLATION, "weights", "best.pt")
model_A_eval = YOLO(BEST_A)

m = model_A_eval.val(
    data=DATA_YAML_HCMC_LOCAL, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name=NAME_ABLATION + "_eval_hcmc", exist_ok=True,
)
A_hcmc_map50 = float(m.box.map50)
A_hcmc_map   = float(m.box.map)
print("===== LO TRINH A (COCO->HCMC) tren TEST HCMC =====")
print(f"mAP@0.5      : {A_hcmc_map50:.4f}")
print(f"mAP@0.5:0.95 : {A_hcmc_map:.4f}")
print(f"Precision    : {float(m.box.mp):.4f}")
print(f"Recall       : {float(m.box.mr):.4f}")

Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.7±0.2 ms, read: 23.1±18.6 MB/s, size: 60.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515/test/labels.cache... 2267 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2267/2267 528.2Mit/s 0.0s
val: /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515/test/images/cam_01_000596_jpg.rf.e27909b3bca621ade420095897f7c5e7.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515/test/images/cam_01_001195_jpg.rf.99126ca4b71bfad3e186294ec9f1f02f.jpg: 1 d

In [9]:
m = model_A_eval.val(
    data=DATA_YAML_COCO_LOCAL, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name=NAME_ABLATION + "_eval_coco", exist_ok=True,
)
A_coco_map50 = float(m.box.map50)
A_coco_map   = float(m.box.map)
print("===== LO TRINH A (COCO->HCMC) tren TEST COCO =====")
print(f"mAP@0.5      : {A_coco_map50:.4f}")
print(f"mAP@0.5:0.95 : {A_coco_map:.4f}")

Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.1 ms, read: 30.4±12.3 MB/s, size: 40.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO_701515/test/labels.cache... 2851 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2851/2851 854.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 179/179 5.2it/s 34.2s
                   all       2851      10683      0.238      0.161     0.0931     0.0457
                   bus        656        973      0.234      0.267      0.156       0.09
                   car       1937       6901      0.492      0.113      0.123     0.0612
             

In [10]:
m = model_A_eval.val(
    data=DATA_YAML_CANTHO, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name=NAME_ABLATION + "_eval_cantho", exist_ok=True,
)
A_cantho_map50 = float(m.box.map50)
A_cantho_map   = float(m.box.map)
print("===== LO TRINH A (COCO->HCMC) tren TEST CANTHO =====")
print(f"mAP@0.5      : {A_cantho_map50:.4f}")
print(f"mAP@0.5:0.95 : {A_cantho_map:.4f}")

Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 70.8±156.8 ms, read: 1.0±1.5 MB/s, size: 40.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Do an AI giao thong/Dataset/CanTho_701515/test/labels.cache... 167 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 167/167 35.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 2.5it/s 4.4s
                   all        167        980        0.7      0.695      0.764      0.539
                   bus          6          7      0.588      0.714      0.693      0.588
                   car        149        494      0.857      0.698      0.873      0.615
              motobyte    

In [11]:

B_hcmc_map50   = 0.953
B_hcmc_map     = 0.686
B_coco_map50   = 0.0843
B_cantho_map50 = 0.750

print("="*78)
print("BANG TONG HOP ABLATION STUDY: COCO->HCMC  vs  COCO->CanTho->HCMC")
print("="*78)
print(f"{'Phep danh gia':40s} {'Lo trinh A':>14s} {'Lo trinh B':>14s}")
print(f"{'(mAP@0.5)':40s} {'COCO->HCMC':>14s} {'COCO->CT->HCMC':>14s}")
print("-"*78)
print(f"{'Tren TEST HCMC (in-domain)':40s} {A_hcmc_map50:>14.4f} {B_hcmc_map50:>14.4f}")
print(f"{'Tren TEST COCO (forgetting)':40s} {A_coco_map50:>14.4f} {B_coco_map50:>14.4f}")
print(f"{'Tren TEST CanTho':40s} {A_cantho_map50:>14.4f} {B_cantho_map50:>14.4f}")
print("="*78)
print()
print("So sanh mAP@0.5:0.95 tren TEST HCMC:")
print(f"  Lo trinh A: {A_hcmc_map:.4f}")
print(f"  Lo trinh B: {B_hcmc_map:.4f}")
print()

# Ket luan tu dong theo dung tinh than thay huong dan
delta = B_hcmc_map50 - A_hcmc_map50
print("--- KET LUAN SO BO ---")
if A_hcmc_map50 < B_hcmc_map50:
    print(f"Lo trinh B (co buoc CanTho) CAO HON lo trinh A tren HCMC test")
    print(f"muc chenh lech mAP@0.5 = {delta:.4f}.")
    print("=> Buoc trung gian CanTho GIUP cai thien hieu nang tren mien dich HCMC,")
    print("   xac nhan gia tri cua chien luoc tinh chinh tuan tu.")
elif A_hcmc_map50 > B_hcmc_map50:
    print(f"Lo trinh A (bo qua CanTho) cao hon lo trinh B tren HCMC test")
    print(f"muc chenh lech mAP@0.5 = {-delta:.4f}.")
    print("=> Can phan tich them: buoc CanTho khong cai thien hieu nang in-domain,")
    print("   nhung can xem xet vai tro cua no ve mat kha nang tong quat (test CanTho).")
else:
    print("Hai lo trinh cho ket qua tuong duong tren HCMC test.")

BANG TONG HOP ABLATION STUDY: COCO->HCMC  vs  COCO->CanTho->HCMC
Phep danh gia                                Lo trinh A     Lo trinh B
(mAP@0.5)                                    COCO->HCMC COCO->CT->HCMC
------------------------------------------------------------------------------
Tren TEST HCMC (in-domain)                       0.9539         0.9530
Tren TEST COCO (forgetting)                      0.0931         0.0843
Tren TEST CanTho                                 0.7635         0.7500

So sanh mAP@0.5:0.95 tren TEST HCMC:
  Lo trinh A: 0.6856
  Lo trinh B: 0.6860

--- KET LUAN SO BO ---
Lo trinh A (bo qua CanTho) cao hon lo trinh B tren HCMC test
muc chenh lech mAP@0.5 = 0.0009.
=> Can phan tich them: buoc CanTho khong cai thien hieu nang in-domain,
   nhung can xem xet vai tro cua no ve mat kha nang tong quat (test CanTho).
